In [29]:
logging.basicConfig(level=logging.INFO)
logger=logging.getLogger()

In [30]:
print(arxiv.__file__)

/workspaces/AI-bootcamp/.venv/lib/python3.12/site-packages/arxiv/__init__.py


In [31]:
import arxiv
from arxiv import Search, SortCriterion, Client
from typing import List,Dict,Any,Optional
from pydantic import BaseModel,Field
from datetime import datetime
import logging
import itertools
from dataclasses import dataclass

In [32]:
@dataclass
class PaperMetadata:
    paper_id: str
    title: str
    authors: List[str]
    abstract: str
    url: str
    published_date: str

In [33]:
def search_arxiv(query: str, max_results: int = 2) -> List[PaperMetadata]:
    """ search arxiv and return papers metadata"""
    logging.info(f"searching arxiv papers for : {query}")
    search = arxiv.Search(
            query = query,
            max_results = max_results,
            sort_by = arxiv.SortCriterion.Relevance
    )
    
    client = Client()
    
    papers = []
    for result in client.results(search):
        paper = PaperMetadata(
            paper_id = result.entry_id.split('/')[-1] ,
            title = result.title,
            authors = [a.name for a in result.authors],
            abstract = result.summary,
            url = result.pdf_url,
            published_date = result.published.strftime("%Y-%m-%d")
        )
        papers.append(paper)
    logging.info(f"found {len(papers)} papers")
    return papers
                                                      
    
        

In [35]:
output = search_arxiv(
            query ="attention mechanism",
            max_results = 2
)

INFO:root:searching arxiv papers for : attention mechanism
INFO:arxiv:Requesting page (first: True, try: 0): https://export.arxiv.org/api/query?search_query=attention+mechanism&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=100
INFO:arxiv:Got first page: 100 of 298740 total results
INFO:root:found 2 papers


In [36]:
print(output)

[PaperMetadata(paper_id='2002.00741v1', title='Déjà vu: A Contextualized Temporal Attention Mechanism for Sequential Recommendation', authors=['Jibang Wu', 'Renqin Cai', 'Hongning Wang'], abstract="Predicting users' preferences based on their sequential behaviors in history is challenging and crucial for modern recommender systems. Most existing sequential recommendation algorithms focus on transitional structure among the sequential actions, but largely ignore the temporal and context information, when modeling the influence of a historical event to current prediction.\n  In this paper, we argue that the influence from the past events on a user's current action should vary over the course of time and under different context. Thus, we propose a Contextualized Temporal Attention Mechanism that learns to weigh historical actions' influence on not only what action it is, but also when and how the action took place. More specifically, to dynamically calibrate the relative input dependence 

In [37]:
import requests
import PyPDF2
import io
def extract_text_from_pdf(pdf_path: str) -> str:
    """ download pdf and extract text """
    logger.info(f"fetching pdf from url :{url}")
    #download
    response=requests.get(url)
    pdf_file = io.BytesIO(response.content)
    #extract text
    reader = PyPDF2.PdfReader(pdf_file)
    text =""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text   
    

In [26]:
from chunking import chunk
class IndexTool:
    """Tool for chunking text and creating embeddings"""
    def __init__(self, openai_client):
        self.name = "index_tool"
        self.client = openai_client
    async def execute(self, text: str, paper_id: str, 
                     chunk_size: int = 1000, overlap: int = 200) -> ToolResult:
        """Chunk text and generate embeddings"""
        try:
            logger.info(f"[IndexTool] Chunking text for paper: {paper_id}")
            
            # Chunk text
            chunks = self._chunk_text(text, paper_id, chunk_size, overlap)
            
            logger.info(f"[IndexTool] Created {len(chunks)} chunks")
            logger.info(f"[IndexTool] Generating embeddings...")
            
            # Generate embeddings
            texts = [chunk.text for chunk in chunks]
            response = self.client.embeddings.create(
                model="text-embedding-3-small",
                input=texts
            )
            
            embeddings = [data.embedding for data in response.data]
            # Attach embeddings to chunks
            chunks_with_embeddings = []
            for chunk, embedding in zip(chunks, embeddings):
                chunks_with_embeddings.append({
                    "chunk": chunk,
                    "embedding": embedding
                })
            
            logger.info(f"[IndexTool] Generated {len(embeddings)} embeddings")
            
            return ToolResult(
                tool_name=self.name,
                success=True,
                data={
                    "chunks": chunks_with_embeddings,
                    "total_chunks": len(chunks)
                }
            )
        except Exception as e:
            logger.error(f"[IndexTool] Error: {e}")
            return ToolResult(
                tool_name=self.name,
                success=False,
                data=None,
                error=str(e)
            )
    

In [38]:
class SemanticSearchTool:
    """Tool for searching similar content in vector database"""
    
    def __init__(self, qdrant_client, openai_client):
        self.name = "semantic_search_tool"
        self.qdrant = qdrant_client
        self.openai = openai_client
    async def execute(self, query: str, top_k: int = 5) -> ToolResult:
        """Search for semantically similar chunks"""
        try:
            logger.info(f"[SemanticSearchTool] Searching for: {query}")
            
            # Embed query
            response = self.openai.embeddings.create(
                model="text-embedding-3-small",
                input=query
            )
            query_embedding = response.data[0].embedding
            
            # Search in Qdrant
            results = self.qdrant.search(
                collection_name="research_papers",
                query_vector=query_embedding,
                limit=top_k
            )
            # Format results
            search_results = []
            for result in results:
                search_results.append({
                    "text": result.payload["text"],
                    "paper_id": result.payload["paper_id"],
                    "chunk_id": result.payload["chunk_id"],
                    "score": result.score
                })
            logger.info(f"[SemanticSearchTool] Found {len(search_results)} results")
            
            return ToolResult(
                tool_name=self.name,
                success=True,
                data={"results": search_results}
            )
        except Exception as e:
            logger.error(f"[SemanticSearchTool] Error: {e}")
            return ToolResult(
                tool_name=self.name,
                success=False,
                data=None,
                error=str(e)
            )

In [ ]:
def main():
    from openai import OpenAI
    from qdrant_client import QdrantClient
    from qdrant_client.models import Distance, VectorParams

    openai_client = OpenAI()
    qdrant_client = QdrantClient(path="./qdrant_data")

    # Create Qdrant collection
    try:
        qdrant_client.create_collection(
            collection_name="research_papers",
            vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
        )
    except:
        pass  # Collection exists
    # Initialize tools
    tools = {
        "search": SearchTool(),
        "fetch": FetchTool(),
        "index": IndexTool(openai_client),
        "semantic_search": SemanticSearchTool(qdrant_client, openai_client),
        "summary": SummaryTool(openai_client)
    }
    
    # Initialize storage
    vector_db = VectorDBManager(qdrant_client)

    # Initialize agent
    agent = ResearchAssistantAgent(
        tools=tools,
        vector_db=vector_db,
        postgres_db=postgres_db,
        openai_client=openai_client
    )